# Анализ датасета "Stroke Prediction": EDA

## 1. Общая информация

| Параметр | Значение |
|----------|----------|
| Источник | [Kaggle (fedesoriano, Stroke Prediction Dataset)](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset) |
| Размер после preprocessing | 4853 строки, 11 колонок |
| Целевая переменная | `stroke` (0 - нет инсульта, 1 - инсульт) |
| Тип задачи | Бинарная классификация |
| Пропуски | 0 (обработаны на этапе preprocessing) |
| Дисбаланс классов | 95.7% / 4.3% (соотношение 22:1) |

## 2. Выполненные преобразования (в рамках EDA)

| Действие | Результат |
|----------|-----------|
| Кодирование категориальных признаков | Бинарные - LabelEncoder, многоклассовые - OneHotEncoder (drop='first') |
| Масштабирование числовых признаков | StandardScaler для `age`, RobustScaler для `avg_glucose_level` и `bmi` |
| Создание новых признаков | `cardio_risk`, `marriage_risk` |
| Удаление неинформативных признаков | `gender`, `smoking_status`, `work_type`, `residence_type`, `senior_work` |
| Отбор признаков на основе корреляции и RF важности | Оставлено 6 финальных признаков |

## 3. Итог после EDA

| Параметр | Значение |
|----------|----------|
| Финальный размер (train) | 2911 строк |
| Размер val / test | 971 / 971 строк |
| Число признаков | 6 |
| Пропуски | 0 |
| Сохранённые версии | `X_train_encoded.csv` (для деревьев), `X_train_scaled.csv` (для линейных моделей) |

---

## 4. Гипотезы перед запуском EDA

Предварительно разделила признаки на три группы по предполагаемой силе влияния. Из-за сильного дисбаланса классов (22:1) даже объективно важные признаки могут выглядеть слабее, чем они есть.

### 4.1. Предположительно сильное влияние

| Признак | Ожидание |
|---------|----------|
| **Возраст** | Самый важный признак. Ожидается сильное смещение распределения при `stroke = 1` вправо. |
| **Гипертония** | Ключевой фактор риска. |
| **Сердечные заболевания** | Ключевой фактор риска, ожидаемое влияние сопоставимо с гипертонией. |
| **ИМТ** | Прокси-признак ожирения и связанных с ним заболеваний (гипертония, диабет). |

### 4.2. Предположительно среднее влияние

| Признак | Ожидание |
|---------|----------|
| **Уровень глюкозы** | Диабет как фактор риска. |
| **Курение** | По медицинским данным увеличивает риск в 2-4 раза. |
| **Тип работы** | Стресс и физические нагрузки как потенциальные факторы. |

### 4.3. Предположительно слабое влияние

| Признак | Ожидание |
|---------|----------|
| **Место жительства** | Город/село - различия в стрессе и экологии. |
| **Пол** | Влияние неравномерно, сильно связано с возрастом. |
| **Семейное положение** | Прокси социальной поддержки, ожидается неоднозначное влияние. |

### 4.4. Фактчекинг и уточнение гипотез

**Курение:** по данным исследований - сильный фактор.

**Семейное положение (`ever_married`):** крайне неоднозначный признак. Эффект может проявиться только в старших возрастных группах. Возможны различные варианты как положительного, так и отрицательного влияния на риск инсульта.

При `ever_married = 1`:
- человек в браке, партнёр жив, есть дети: риск инсульта снижается, возрастает вероятность благоприятного восстановления;
- человек был в браке, разведён, утрата партнёра, нет детей: увеличение риска, уменьшение вероятности благополучного восстановления.

---

## 5. Анализ исходных признаков

### 5.1. Числовые признаки

#### Возраст

- Гистограмма близка к нормальному распределению (медиана ~45 лет, пик ~50 лет).
- При `stroke = 0` - широкий нормальный колокол.
- При `stroke = 1` - бимодальное распределение с пиками ~58 и ~77-78 лет.
- **Вывод:** два пика могут указывать на разные группы риска.

#### Уровень глюкозы

- Бимодальное распределение: первый пик ~90 мг/дл (норма), второй ~215-220 мг/дл (диабет).
- При `stroke = 1` плотность первого пика в 2 раза ниже, второго - в 2 раза выше.
- **Вывод:** признак чётко выделяет группу диабетиков с повышенным риском. Удалять «выбросы» нельзя.

#### ИМТ

- Распределение близко к нормальному (пик ~26-27), небольшая правосторонняя скошенность.
- При `stroke = 1` левая часть колокола уже, риск начинает расти уже при ИМТ >25-27.
- **Вывод:** умеренная, но устойчивая связь.

### 5.2. Бинарные признаки

| Признак | Доля | stroke при 0 | stroke при 1 | Lift |
|---------|------|--------------|--------------|------|
| `hypertension` | 9.5% | 3.5% | 12.2% | 3.5x |
| `heart_disease` | 5.0% | 3.7% | 16.6% | 4.5x |

Оба признака - сильные предикторы. Сердечные заболевания оказались более сильным фактором, несмотря на меньшую распространённость.  
**Решение:** после всех этапов анализа объединены в `cardio_risk` (ИЛИ).

### 5.3. Семейное положение (`ever_married`)

- Без учёта возраста: у женатых риск выше (5.7% vs 1.6%) - обманчивый вывод.
- Стратификация по возрасту:
  - **<40:** различия минимальны, маленькая выборка.
  - **40-60:** разница незначительна.
  - **60+:** у неженатых **21.1%** инсультов, у женатых **10.3%**.
- Эффект не зависит от пола.
- Группа неженатых пожилых мала (49 чел.), но сигнал очень сильный.
- **Решение:** создан составной признак `marriage_risk`.

### 5.4. Пол

При анализе по возрасту устойчивых различий между мужчинами и женщинами не обнаружено. Дополнительные комбинации с другими признаками показывали идентичные результаты.  
**Решение:** признак исключён.

### 5.5. Курение (`smoking_status`)

- У `formerly_smoked` риск 5.9%, у `smokes` - 4.8%, у `never_smoked` - 4.7%.
- В группе 70+ у `never_smoked` самый высокий риск (19.9%).
- Корреляция с целью: **-0.059**.
- Дополнительные комбинации с другими признаками существенного сигнала не дали.
- **Решение:** признак исключён из-за противоречивого и слабого сигнала.

### 5.6. Тип работы (`work_type`)

- Самый высокий риск у `self-employed` (6.6%).
- Корреляция с целью: **0.077** (слабая).
- Производные (`is_high_risk_work`, `senior_stress`) либо не улучшали сигнал, либо сильно коррелировали с возрастом.
- **Решение:** признак и его производные исключены.

### 5.7. Тип проживания (`residence_type`)

- Разница между городом и селом практически отсутствует (4.4% vs 4.2%).
- Корреляция с целью: **0.007**.
- Дополнительные комбинации с другими признаками существенного сигнала не дали.
- **Решение:** признак исключён.

---

## 6. Создание и анализ новых признаков

### 6.1. `cardio_risk` (ИЛИ для гипертонии и сердечных заболеваний)

- Доля группы: 13.2%.
- Риск инсульта: 13.0% vs 3.0% (**Lift = 4.4x**).
- Корреляция с целью: **0.167** (выше, чем у исходных).
- **Решение:** оставлен.

### 6.2. `marriage_risk` (возраст + `ever_married`)

- Категории: `young` (возраст ≤ 60), `senior_married` (возраст > 60 и `ever_married = 1`), `senior_unmarried` (возраст > 60 и `ever_married = 0`).
- Риски: `young` - 1.9%, `senior_married` - 10.3%, `senior_unmarried` - **21.1%**.
- **Решение:** оставлен с оговорками (One-Hot кодирование, `young` - базовая категория). Влияние возраста может превышать влияние самого `marriage_risk`. На этапе моделирования будет проверена версия модели без этого признака.

### 6.3. Прочие производные

- `high_risk_metabolic` (глюкоза > 126 и ИМТ > 30): корреляция 0.085 (хуже исходных).
- `senior_work` и варианты: корреляция с возрастом >0.67 - избыточны.
- **Решение:** все исключены.

---

## 7. Корреляционный анализ и отбор признаков

### 7.1. Матрица корреляции (ключевые наблюдения)

- Исходные признаки (`age`, `avg_glucose_level`, `bmi`, `cardio_risk`) не имеют критических корреляций (max 0.36).
- `senior_work` и `marriage_risk_senior_married` сильно коррелируют с `age` (0.70-0.74).
- `marriage_risk_senior_unmarried` имеет низкую корреляцию с `age` (0.20) - **уникальный сигнал**.

### 7.2. Удаление по корреляции

Удалены: `senior_work`, `senior_work_1`, `senior_work_2`.  
Оставлены: `age`, `avg_glucose_level`, `bmi`, `cardio_risk`, `marriage_risk_senior_married`, `marriage_risk_senior_unmarried`.

---

## 8. Оценка важности (Random Forest)

**Параметры:** `n_estimators=100`, `class_weight='balanced'`.

| Признак | Важность | Доля |
|---------|----------|------|
| `age` | 0.3817 | 38.2% |
| `avg_glucose_level` | 0.2559 | 25.6% |
| `bmi` | 0.2532 | 25.3% |
| `marriage_risk_senior_married` | 0.0542 | 5.4% |
| `cardio_risk` | 0.0426 | 4.3% |

**Накопленная важность:**
- 50% - 2 признака
- 80% - 3 признака
- 90% - 4 признака
- 95% - 5 признаков

`marriage_risk_senior_unmarried` имеет важность ~0.01, но **сохранён** из-за сильного увеличения риска (21.1%).

**Финальный набор: 6 признаков.**

---

## 9. Финальная подготовка данных

### 9.1. Масштабирование

| Признак | Метод | До (μ ± σ) | После (μ ± σ) |
|---------|-------|------------|---------------|
| `age` | StandardScaler | 43.6 ± 22.2 | 0.00 ± 1.00 |
| `avg_glucose_level` | RobustScaler | 105.0 ± 44.7 | 0.38 ± 1.23 |
| `bmi` | RobustScaler | 28.9 ± 7.4 | 0.08 ± 0.80 |

### 9.2. Сохранённые файлы

- `X_train_encoded.csv` для деревьев
- `X_train_scaled.csv` для линейных моделей
- `X_val_final.csv`, `X_test_final.csv`
- `scalers.pkl`, `encoders.pkl`, `metadata.json`

### 9.3. Итоговая статистика

| Параметр | Значение |
|----------|----------|
| Финальных признаков | 6 |
| Пропуски | 0 |
| Масштабирование | ✓ |
| Кодирование | ✓ |

---

## 10. Рекомендации для моделирования

- Для деревьев: `X_train_encoded.csv`
- Для линейных моделей: `X_train_scaled.csv`
- Учёт дисбаланса: `class_weight='balanced'` или SMOTE
- Метрики: **ROC-AUC**, **F1-score** (не Accuracy)
- Сравнение моделей на валидации, финальная оценка - на тесте

---

**EDA завершён. Данные готовы к обучению моделей.**